# Phase 2

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)

SEED = 42
np.random.seed(SEED)

sns.set_theme(
    style="whitegrid",
    context="notebook",
    palette="colorblind"
)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "font.size": 10
})


# LOCAL DIRECTORIES


# Change this if your actual folder name is different.
REAL_DATA_DIR = Path("Real cohort Data")

REAL_DATA_CANDIDATES = [
    REAL_DATA_DIR / "Phase_2_data_Prep.csv",
    Path("Real cohort Data folder") / "Phase_2_data_Prep.csv",
    Path("Phase_2_data_Prep.csv")
]

REAL_DATA_PATH = next(
    (p for p in REAL_DATA_CANDIDATES if p.is_file()),
    None
)

if REAL_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate Phase_2_data_Prep.csv. "
        "Update REAL_DATA_DIR to your local data folder."
    )

# Use the actual folder containing the CSV.
REAL_DATA_DIR = REAL_DATA_PATH.parent

# Phase 1 model locations from the previous notebook.
PHASE1_MODEL_DIR = Path(
    "synthetic_nsc_data/phase1_model_outputs/models"
)

PHASE1_MODEL_CANDIDATES = [
    PHASE1_MODEL_DIR / "phase1_multistage_academic_risk_model.joblib",
    PHASE1_MODEL_DIR / "phase1_best_multistage_risk_model.joblib"
]

PHASE1_MODEL_PATH = next(
    (p for p in PHASE1_MODEL_CANDIDATES if p.is_file()),
    None
)

if PHASE1_MODEL_PATH is None:
    raise FileNotFoundError(
        "Phase 1 trained model was not found. "
        "Set PHASE1_MODEL_DIR to the directory containing "
        "your Phase 1 .joblib model."
    )

PHASE1_RESULTS_DIR = Path(
    "synthetic_nsc_data/phase1_model_outputs/results"
)


# PHASE 2 OUTPUT DIRECTORIES


OUTPUT_DIR = REAL_DATA_DIR / "Phase_2_Outputs"

METRIC_DIR = OUTPUT_DIR / "metrics"
FIGURE_DIR = OUTPUT_DIR / "figures"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"

for directory in [
    OUTPUT_DIR,
    METRIC_DIR,
    FIGURE_DIR,
    PREDICTION_DIR,
    MODEL_DIR,
    REPORT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# Maintain the original classification threshold.
# It will be replaced by the threshold saved in Phase 1.
DEFAULT_THRESHOLD = 0.50

# Bootstrap resampling for uncertainty intervals.
N_BOOTSTRAP = 1000

# Minimum subgroup size for descriptive fairness reporting.
MIN_SUBGROUP_N = 20

# Run real-data model refitting after frozen evaluation.
RUN_REAL_REFIT = True


def save_figure(fig, filename):
    """Save and display a publication-quality figure."""
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


print("Phase 2 configured successfully.")
print("Real data:", REAL_DATA_PATH.resolve())
print("Phase 1 model:", PHASE1_MODEL_PATH.resolve())
print("Outputs:", OUTPUT_DIR.resolve())

# Load the Phase 1 Synthetic Trained Model

The final synthetic-trained model is loaded using joblib

The saved model contains a separate predictor for each of the four
academic risk checkpoints

No retraining or recalibration is performed during this step

This ensures that the first Phase 2 evaluation measures the transferability
of the model developed using synthetic data

In [ ]:
phase1_bundle = joblib.load(PHASE1_MODEL_PATH)

if not isinstance(phase1_bundle, dict):
    raise TypeError(
        "The saved Phase 1 artifact must be a dictionary "
        "containing the stage models and feature definitions."
    )

STAGE_ORDER = phase1_bundle["stage_order"]
STAGE_FEATURES = phase1_bundle["stage_features"]

# Support both saved naming conventions from Phase 1.
source_models = phase1_bundle.get(
    "models",
    phase1_bundle.get("stage_models")
)

selected_model_names = phase1_bundle.get(
    "selected_models",
    phase1_bundle.get("selected_model_by_stage", {})
)

if source_models is None:
    raise KeyError(
        "The Phase 1 artifact does not contain "
        "'models' or 'stage_models'."
    )

THRESHOLD = float(
    phase1_bundle.get(
        "classification_threshold",
        DEFAULT_THRESHOLD
    )
)

STAGE_SLUGS = {
    "Pre University": "pre_university",
    "1st Semester": "first_semester",
    "Early 2nd Semester": "early_second_semester",
    "2nd Semester": "second_semester"
}

S1_MODULES = phase1_bundle.get(
    "semester_1_modules", []
)

S2_MODULES = phase1_bundle.get(
    "semester_2_modules", []
)

for stage in STAGE_ORDER:
    if stage not in source_models:
        raise KeyError(f"Missing saved model for {stage}")

    if stage not in STAGE_FEATURES:
        raise KeyError(f"Missing feature list for {stage}")


print("Phase 1 model loaded successfully.")
print(f"Classification threshold: {THRESHOLD:.2f}")

for stage in STAGE_ORDER:
    print(
        f"{stage:<25} "
        f"Model: {selected_model_names.get(stage, 'Saved estimator'):<24} "
        f"Features: {len(STAGE_FEATURES[stage])}"
    )

# Load and Validate Real Student Data

The real dataset was prepared separately and contains 1st year BSc Computer
Science student records.

The file includes:

- student and institutional information
- NSC academic preparation
- university module marks and results
- progression credit information
- academic risk outcomes 
- engineered checkpoint features

The `cohort_year` field is standardised to `entry_year` for compatibility
with the Phase 1 workflow

The academic-risk target remains:

- 1: Not readmitted under the faculty progression rules
- 0: Readmitted under the faculty progression rules

The target is verified against the readmission indicator before evaluation

In [ ]:
real_raw = pd.read_csv(
    REAL_DATA_PATH,
    low_memory=False
)

real = real_raw.copy()

required = [
    "student_id",
    "cohort_year",
    "academic_risk",
    "readmitted_to_second_year"
]

missing = [
    col for col in required
    if col not in real.columns
]

if missing:
    raise ValueError(
        f"Required real-data columns missing: {missing}"
    )

# Match the Phase 1 naming convention.
real["entry_year"] = pd.to_numeric(
    real["cohort_year"],
    errors="raise"
).astype(int)

real["academic_risk"] = pd.to_numeric(
    real["academic_risk"],
    errors="raise"
).astype(int)

real["readmitted_to_second_year"] = pd.to_numeric(
    real["readmitted_to_second_year"],
    errors="raise"
).astype(int)


# Data integrity checks


if real["student_id"].duplicated().any():
    raise ValueError(
        "Duplicate student_id values detected. "
        "Investigate duplicates before modelling."
    )

if not real["academic_risk"].isin([0, 1]).all():
    raise ValueError("Academic risk must be binary.")

if not (
    real["academic_risk"]
    == (1 - real["readmitted_to_second_year"])
).all():
    raise ValueError(
        "Academic-risk and readmission labels are inconsistent."
    )

# Confirm the prepared credit values are internally consistent.
if {
    "planned_credits",
    "passed_credits",
    "credit_completion_rate"
}.issubset(real.columns):

    calculated_rate = (
        real["passed_credits"]
        / real["planned_credits"].replace(0, np.nan)
    )

    credit_difference = (
        calculated_rate
        - real["credit_completion_rate"]
    ).abs()

    print(
        "Rows with credit-rate discrepancies:",
        int((credit_difference > 1e-6).sum())
    )

print("\nDataset shape:", real.shape)
print("Years:", sorted(real["entry_year"].unique()))
print("Academic-risk prevalence:", round(real["academic_risk"].mean(), 4))

cohort_summary = (
    real.groupby("entry_year")["academic_risk"]
    .agg(["size", "sum", "mean"])
    .reset_index()
    .rename(columns={
        "size": "students",
        "sum": "at_risk_students",
        "mean": "risk_rate"
    })
)

cohort_summary["risk_rate_percent"] = (
    cohort_summary["risk_rate"] * 100
)

display(cohort_summary.round(3))

cohort_summary.to_csv(
    METRIC_DIR / "01_real_cohort_summary.csv",
    index=False
)

# Save the original column-completeness audit.
raw_completeness = pd.DataFrame({
    "column": real_raw.columns,
    "non_missing": real_raw.notna().sum().values,
    "missing": real_raw.isna().sum().values,
    "missing_percent": (
        real_raw.isna().mean().values * 100
    )
})

raw_completeness.to_csv(
    REPORT_DIR / "original_real_data_completeness.csv",
    index=False
)

# Descriptive Analysis of the Real Cohorts

The number of available students and observed academic risk rates are
visualised by year.

This allows differences between cohorts to be examined before model
evaluation.

Any differences in risk prevalence are treated as descriptive observations
rather than proof that the underlying academic population changed by the
same magnitude.

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(13, 5)
)

sns.barplot(
    data=cohort_summary,
    x="entry_year",
    y="students",
    color=sns.color_palette("deep")[0],
    ax=axes[0]
)

axes[0].set_title("Real Student Cohort Sizes")
axes[0].set_xlabel("Entry Year")
axes[0].set_ylabel("Number of Students")

sns.barplot(
    data=cohort_summary,
    x="entry_year",
    y="risk_rate_percent",
    color=sns.color_palette("deep")[1],
    ax=axes[1]
)

axes[1].set_title("Academic Risk by Cohort")
axes[1].set_xlabel("Entry Year")
axes[1].set_ylabel("Academic Risk (%)")
axes[1].set_ylim(0, 100)

save_figure(
    fig,
    "01_real_cohort_distribution.png"
)

# Feature Alignment Between Synthetic and Real Data

The synthetic trained models require the same feature names and definitions
during Phase 2 as they used during Phase 1.

This section performs limited, defensible harmonisation:

- standardises school language categories
- converts registration and Boolean flags
- derives an NSC average when all seven required school marks are available
- derives prior Computer Science exposure from named school subjects
- reconstructs selected mathematical features where possible
- preserves genuinely unavailable variables as missing

Missing variables are not replaced with simulated real student information

The frozen Phase 1 pipelines perform their originally fitted imputation
scaling and encoding during prediction.

In [ ]:
aligned = real.copy()
original_columns = set(real_raw.columns)


# School-language harmonisation


LANGUAGE_MAP = {
    "seSotho sa Lebowa": "Sepedi",
    "seSotho": "Sesotho",
    "Isizulu": "isiZulu",
    "Isixhosa": "isiXhosa",
    "Siswati": "siSwati"
}


def standardise_home_language(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in ["Language unknown", "Other language", ""]:
        return np.nan

    value = LANGUAGE_MAP.get(value, value)

    if not value.endswith(" HL"):
        value += " HL"

    return value


if "home_language" in aligned:
    aligned["home_language"] = (
        aligned["home_language"]
        .map(standardise_home_language)
    )


def standardise_school_subject(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    value = value.replace(
        " Home Language", " HL"
    ).replace(
        " 1st Language", " HL"
    ).replace(
        " 1st Add Language", " FAL"
    )

    replacements = {
        "Isizulu": "isiZulu",
        "Isixhosa": "isiXhosa",
        "Siswati": "siSwati",
        "Sesotho": "Sesotho",
        "Sepedi": "Sepedi"
    }

    for old, new in replacements.items():
        if value.startswith(old):
            value = new + value[len(old):]
            break

    return value


for column in [
    "subject_home_language",
    "subject_fal"
]:
    if column in aligned:
        aligned[column] = (
            aligned[column]
            .map(standardise_school_subject)
        )


# The real file records the subject name; use that same
# observed subject for the corresponding Phase 1 field.
if "first_additional_lang" not in aligned.columns:
    if "subject_fal" in aligned.columns:
        aligned["first_additional_lang"] = aligned["subject_fal"]


# Harmonise equivalent NSC elective subject names.
ELECTIVE_MAP = {
    "Computer Applic Technology": "CAT",
    "Physical Science": "Physical Sciences",
    "Engineering Graphics & Design": "EGD"
}

for column in [
    "subject_elective_1",
    "subject_elective_2",
    "subject_elective_3"
]:
    if column in aligned:
        aligned[column] = aligned[column].replace(ELECTIVE_MAP)



# Prior Computer Science exposure


elective_columns = [
    c for c in [
        "subject_elective_1",
        "subject_elective_2",
        "subject_elective_3"
    ]
    if c in aligned.columns
]

if elective_columns and "has_prior_cs_subject" not in aligned:

    subjects = aligned[elective_columns].fillna("")

    cs_pattern = (
        r"Information Technology|Computer Science|"
        r"Computer Applications Technology|CAT"
    )

    has_cs = subjects.apply(
        lambda col: col.str.contains(
            cs_pattern,
            case=False,
            regex=True
        )
    ).any(axis=1)

    all_subjects_known = subjects.ne("").all(axis=1)

    aligned["has_prior_cs_subject"] = (
        has_cs.where(all_subjects_known, np.nan)
    )



# Derive NSC average when a complete set is available


nsc_mark_columns = [
    "mark_home_language",
    "mark_fal",
    "mark_life_orientation",
    "mark_maths_or_ml",
    "mark_elective_1",
    "mark_elective_2",
    "mark_elective_3"
]

if all(c in aligned for c in nsc_mark_columns):

    school_marks = aligned[nsc_mark_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    derived_average = school_marks.mean(axis=1)

    # Avoid manufacturing complete school averages
    # from incomplete subject records.
    derived_average = derived_average.where(
        school_marks.notna().sum(axis=1) == 7
    )

    if "overall_average" not in aligned:
        aligned["overall_average"] = derived_average



# Reproduce the Phase 1 entry academic feature


entry_mark_columns = [
    c for c in [
        "mark_maths_or_ml",
        "english_mark",
        "physical_science_mark",
        "prior_cs_mark",
        "overall_average"
    ]
    if c in aligned.columns
]

if entry_mark_columns:
    aligned["entry_academic_mean"] = (
        aligned[entry_mark_columns]
        .apply(pd.to_numeric, errors="coerce")
        .mean(axis=1)
    )



# Reconstruct missing module aggregates using Phase 1 mapping


def module_columns(modules, suffix):
    return [
        f"{module}_{suffix}"
        for module in modules
        if f"{module}_{suffix}" in aligned.columns
    ]


s1_final_cols = module_columns(S1_MODULES, "final_mark")
s1_ca_cols = module_columns(S1_MODULES, "semester_mark")
s2_ca_cols = module_columns(S2_MODULES, "semester_mark")

def fill_engineered(column, values):
    if column not in aligned:
        aligned[column] = values
    else:
        aligned[column] = aligned[column].fillna(values)


if s1_final_cols:
    marks = aligned[s1_final_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    fill_engineered(
        "s1_final_min",
        marks.min(axis=1)
    )

    fill_engineered(
        "s1_final_below_50_count",
        marks.lt(50).sum(axis=1)
    )

    fill_engineered(
        "s1_final_observed_count",
        marks.notna().sum(axis=1)
    )


if s1_ca_cols:
    marks = aligned[s1_ca_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    fill_engineered(
        "s1_ca_mean",
        marks.mean(axis=1)
    )

    fill_engineered(
        "s1_ca_min",
        marks.min(axis=1)
    )

    fill_engineered(
        "s1_ca_below_50_count",
        marks.lt(50).sum(axis=1)
    )

    fill_engineered(
        "s1_ca_observed_count",
        marks.notna().sum(axis=1)
    )


if s2_ca_cols:
    marks = aligned[s2_ca_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    fill_engineered(
        "s2_ca_mean",
        marks.mean(axis=1)
    )

    fill_engineered(
        "s2_ca_min",
        marks.min(axis=1)
    )

    fill_engineered(
        "s2_ca_below_50_count",
        marks.lt(50).sum(axis=1)
    )

    fill_engineered(
        "s2_ca_observed_count",
        marks.notna().sum(axis=1)
    )



# Align Boolean data types


boolean_features = {
    "has_mathematics",
    "has_ap_mathematics",
    "has_ap_english",
    "has_prior_cs_subject",
    "bachelor_eligible",
    "passed_nsc",
    "progressed_learner",
    "semester2_permitted"
}

all_stage_features = list(dict.fromkeys(
    feature
    for stage in STAGE_ORDER
    for feature in STAGE_FEATURES[stage]
))

for feature in all_stage_features:
    if feature.endswith("_registered"):
        boolean_features.add(feature)


def standardise_boolean(series):
    mapping = {
        "1": True,
        "1.0": True,
        "true": True,
        "yes": True,
        "0": False,
        "0.0": False,
        "false": False,
        "no": False
    }

    result = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
    )

    return result.astype(object)


for feature in boolean_features:
    if feature in aligned.columns:
        aligned[feature] = standardise_boolean(
            aligned[feature]
        )



# Create genuinely missing inputs as NaN


missing_phase1_columns = []

for feature in all_stage_features:
    if feature not in aligned.columns:
        aligned[feature] = np.nan
        missing_phase1_columns.append(feature)


print("Real feature alignment completed.")
print("Unique Phase 1 features:", len(all_stage_features))
print("Unavailable feature columns:", len(missing_phase1_columns))

if missing_phase1_columns:
    print("\nUnavailable fields:")
    print(missing_phase1_columns)

# Feature Availability and Information Leakage Audit

A separate report is created for each academic checkpoint

The report records:

- Features required by the saved synthetic model
- Whether each feature was available in the original real CSV
- Whether it was derived from available information
- Missing values
- The proportion of real student records where the field is available.

The pipeline also rejects final annual outcome variables and information
that would not be permissible at the corresponding checkpoint

The purpose is to distinguish a genuine loss of model performance from
potential differences in data capture and availability

In [ ]:
BANNED_FEATURES = {
    "academic_risk",
    "readmitted_to_second_year",
    "first_year_status",
    "passed_credits",
    "credit_completion_rate",
    "annual_credit_weighted_average",
    "annual_weighted_mark_points",
    "passed_modules",
    "dropped_credits",
    "target_year2_rate",
    "synthetic_year2_rate",
    "mark_calibration_shift"
}


def validate_stage_timing(stage, features):

    invalid = set(features).intersection(
        BANNED_FEATURES
    )

    if stage == "Pre University":
        invalid.update([
            feature for feature in features
            if (
                feature.startswith(("s1_", "s2_"))
                or feature.endswith((
                    "_semester_mark",
                    "_exam_mark",
                    "_final_mark",
                    "_result"
                ))
                or feature == "semester2_permitted"
            )
        ])

    if stage == "1st Semester":
        invalid.update([
            feature for feature in features
            if (
                feature.endswith((
                    "_exam_mark",
                    "_final_mark",
                    "_result"
                ))
                or feature.startswith("s2_")
                or feature == "semester2_permitted"
            )
        ])

    if stage in ["Early 2nd Semester", "2nd Semester"]:
        invalid.update([
            feature for feature in features
            if any(
                feature.startswith(f"{module}_")
                and feature.endswith((
                    "_exam_mark",
                    "_final_mark",
                    "_result"
                ))
                for module in S2_MODULES
            )
        ])

    if stage == "Early 2nd Semester":
        invalid.update([
            feature for feature in features
            if any(
                feature.startswith(f"{module}_")
                and feature.endswith("_semester_mark")
                for module in S2_MODULES
            )
            or feature.startswith("s2_ca_")
        ])

    if invalid:
        raise ValueError(
            f"Temporal leakage detected in {stage}: "
            f"{sorted(invalid)}"
        )


audit_rows = []

for stage in STAGE_ORDER:

    features = STAGE_FEATURES[stage]

    validate_stage_timing(stage, features)

    for feature in features:

        if feature in original_columns:
            origin = "Original CSV"
        elif feature in missing_phase1_columns:
            origin = "Unavailable"
        else:
            origin = "Derived / harmonised"

        audit_rows.append({
            "stage": stage,
            "feature": feature,
            "origin": origin,
            "non_missing": int(aligned[feature].notna().sum()),
            "missing": int(aligned[feature].isna().sum()),
            "availability_percent": (
                aligned[feature].notna().mean() * 100
            )
        })


feature_audit_df = pd.DataFrame(audit_rows)

feature_audit_df.to_csv(
    REPORT_DIR / "phase2_feature_alignment_audit.csv",
    index=False
)

stage_coverage = (
    feature_audit_df
    .groupby("stage", as_index=False)
    .agg(
        required_features=("feature", "count"),
        mean_availability_percent=(
            "availability_percent",
            "mean"
        ),
        unavailable_features=(
            "origin",
            lambda s: int((s == "Unavailable").sum())
        )
    )
)

stage_coverage["stage"] = pd.Categorical(
    stage_coverage["stage"],
    categories=STAGE_ORDER,
    ordered=True
)

stage_coverage = stage_coverage.sort_values("stage")

display(stage_coverage.round(2))

stage_coverage.to_csv(
    REPORT_DIR / "phase2_stage_coverage.csv",
    index=False
)


fig, ax = plt.subplots(figsize=(10, 5))

sns.barplot(
    data=stage_coverage,
    x="stage",
    y="mean_availability_percent",
    color=sns.color_palette("deep")[0],
    ax=ax
)

ax.set_title("Real-Data Feature Availability by Checkpoint")
ax.set_xlabel("Prediction Checkpoint")
ax.set_ylabel("Average Observed Feature Availability (%)")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=15)

save_figure(
    fig,
    "02_feature_availability_by_stage.png"
)

print("Temporal leakage checks passed.")

# Evaluation Metrics

The frozen Phase 1 models are assessed using the same main metrics as in
the synthetic data experiment:

- Accuracy
- Balanced Accuracy
- Precision
- Recall / Sensitivity
- Specificity
- F1 Score
- ROC AUC
- Average Precision
- Brier Score
- Log Loss

False negatives are especially important because they represent genuinely
at-risk students who were not identified

Brier score and log loss assess probability quality rather than only binary
classification performance

A threshold of 0.50 is retained for comparability with Phase 1

In [ ]:
def evaluate_model(y_true, probabilities, threshold=THRESHOLD):

    y_true = np.asarray(y_true).astype(int)
    probabilities = np.asarray(probabilities, dtype=float)

    probabilities = np.clip(
        probabilities,
        1e-7,
        1 - 1e-7
    )

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    has_both_classes = len(np.unique(y_true)) == 2

    results = {
        "n_students": len(y_true),
        "n_at_risk": int(y_true.sum()),
        "risk_prevalence": float(y_true.mean()),

        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),

        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "specificity": (
            tn / (tn + fp) if (tn + fp) > 0 else np.nan
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),

        "roc_auc": (
            roc_auc_score(y_true, probabilities)
            if has_both_classes else np.nan
        ),

        "average_precision": (
            average_precision_score(y_true, probabilities)
            if y_true.sum() > 0 else np.nan
        ),

        "brier": brier_score_loss(
            y_true, probabilities
        ),

        "log_loss": log_loss(
            y_true,
            probabilities,
            labels=[0, 1]
        ),

        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }

    return results


def assign_risk_band(probabilities):

    probabilities = np.asarray(probabilities)

    return pd.cut(
        probabilities,
        bins=[-0.000001, 0.2, 0.4, 0.6, 0.8, 1.000001],
        labels=[
            "Low",
            "Medium Low",
            "Medium",
            "Medium High",
            "High"
        ]
    ).astype(str)

# External Evaluation of Synthetic Trained Models

The 4 Phase 1 models are applied without retraining to the prepared real
student dataset

Each stage uses only its approved features

For every stage the experiment records:

- Student risk probabilities
- Predicted risk categories
- Classification outcomes
- Performance metrics for all real students
- Performance metrics for each real cohort year

The results measure how the synthetic trained models perform when
transferred to real 1st year institutional records

Because the real readmission labels were prepared using the faculty rules,
the measured outcome is rule based academic progression risk

In [ ]:
real_targets = aligned["academic_risk"].to_numpy()

frozen_probabilities = {}
frozen_metric_rows = []

prediction_df = aligned[
    ["student_id", "entry_year", "academic_risk"]
].copy()

for stage in STAGE_ORDER:

    print(f"\nEvaluating frozen Phase 1 model: {stage}")

    features = STAGE_FEATURES[stage]
    model = source_models[stage]

    X_real = aligned[features].copy()

    try:
        probabilities = model.predict_proba(X_real)[:, 1]

    except Exception as exc:
        raise RuntimeError(
            f"Frozen Phase 1 prediction failed at {stage}. "
            "Check the saved model schema and Phase 2 "
            "feature-alignment report."
        ) from exc

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"Non-finite probabilities returned at {stage}."
        )

    frozen_probabilities[stage] = probabilities

    slug = STAGE_SLUGS[stage]

    prediction_df[
        f"{slug}_risk_probability"
    ] = probabilities

    prediction_df[
        f"{slug}_risk_band"
    ] = assign_risk_band(probabilities)

    prediction_df[
        f"{slug}_predicted_class"
    ] = (probabilities >= THRESHOLD).astype(int)

  
    # Evaluation across all real cohorts
  

    metrics = evaluate_model(
        real_targets,
        probabilities
    )

    frozen_metric_rows.append({
        "model_version": "Frozen synthetic",
        "stage": stage,
        "cohort": "All Real Cohorts",
        "selected_algorithm": selected_model_names.get(stage),
        **metrics
    })

    print(
        f"ROC AUC: {metrics['roc_auc']:.4f} | "
        f"Recall: {metrics['recall']:.4f} | "
        f"F1: {metrics['f1']:.4f} | "
        f"Brier: {metrics['brier']:.4f}"
    )

  
    # Evaluation by real cohort year
  

    for year in sorted(aligned["entry_year"].unique()):

        mask = (
            aligned["entry_year"].to_numpy() == year
        )

        year_metrics = evaluate_model(
            real_targets[mask],
            probabilities[mask]
        )

        frozen_metric_rows.append({
            "model_version": "Frozen synthetic",
            "stage": stage,
            "cohort": str(year),
            "selected_algorithm": selected_model_names.get(stage),
            **year_metrics
        })


frozen_metrics_df = pd.DataFrame(
    frozen_metric_rows
)

frozen_overall_df = (
    frozen_metrics_df[
        frozen_metrics_df["cohort"] == "All Real Cohorts"
    ]
    .copy()
)

frozen_overall_df["stage"] = pd.Categorical(
    frozen_overall_df["stage"],
    categories=STAGE_ORDER,
    ordered=True
)

frozen_overall_df = frozen_overall_df.sort_values("stage")

display(
    frozen_overall_df[
        [
            "stage",
            "selected_algorithm",
            "n_students",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "average_precision",
            "brier",
            "log_loss"
        ]
    ].round(4)
)

frozen_metrics_df.to_csv(
    METRIC_DIR / "02_frozen_synthetic_models_real_evaluation.csv",
    index=False
)

frozen_overall_df.to_csv(
    METRIC_DIR / "03_frozen_models_overall_metrics.csv",
    index=False
)

prediction_df.to_csv(
    PREDICTION_DIR / "01_real_student_frozen_risk_predictions.csv",
    index=False
)

print("\nFrozen synthetic-model evaluation completed.")

# Comparison of Academic-Risk Performance Across Checkpoints

The performance of the selected Phase 1 models is visualised using real
institutional data

The objective is to establish whether additional academic information
improves model performance across the 4 checkpoints

ROC AUC, recall, precision and F1 score are displayed together

These are external evaluation results, not the synthetic training metrics

In [ ]:
metrics_to_plot = [
    "roc_auc",
    "recall",
    "precision",
    "f1"
]

plot_df = frozen_overall_df.melt(
    id_vars="stage",
    value_vars=metrics_to_plot,
    var_name="Metric",
    value_name="Score"
)

plot_df["Metric"] = plot_df["Metric"].replace({
    "roc_auc": "ROC AUC",
    "recall": "Recall",
    "precision": "Precision",
    "f1": "F1 Score"
})

fig, ax = plt.subplots(figsize=(13, 6))

sns.barplot(
    data=plot_df,
    x="stage",
    y="Score",
    hue="Metric",
    ax=ax
)

ax.set_title(
    "Frozen Synthetic Models: Performance on Real Cohort Data"
)
ax.set_xlabel("Prediction Checkpoint")
ax.set_ylabel("Performance Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)

ax.legend(
    title="Evaluation Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

save_figure(
    fig,
    "03_real_model_performance_by_checkpoint.png"
)

# ROC Curve Analysis

ROC curves demonstrate how well each synthetic trained model distinguishes
at risk students from students who satisfy the progression requirements

A separate ROC curve is generated for each checkpoint

The area under each curve is reported in the legend of the figure

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 11)
)

for ax, stage in zip(axes.flat, STAGE_ORDER):

    probabilities = frozen_probabilities[stage]

    fpr, tpr, _ = roc_curve(
        real_targets,
        probabilities
    )

    auc_score = roc_auc_score(
        real_targets,
        probabilities
    )

    sns.lineplot(
        x=fpr,
        y=tpr,
        ax=ax,
        linewidth=2.2,
        label=f"ROC AUC = {auc_score:.3f}"
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="gray",
        label="Random classifier"
    )

    ax.set_title(stage)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="lower right")

fig.suptitle(
    "ROC Curves of Synthetic Trained Models on Real Data",
    fontsize=16,
    y=1.01
)

save_figure(
    fig,
    "04_real_data_roc_curves.png"
)

# Precision-Recall Analysis

Precision-recall curves are used to assess the models' ability to identify
at risk students without generating excessive false-positive alerts.

Average precision is calculated for every checkpoint.

This analysis is particularly important for a resource constrained
early warning system where not all students can receive intensive
intervention simultaneously.

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 11)
)

for ax, stage in zip(axes.flat, STAGE_ORDER):

    probabilities = frozen_probabilities[stage]

    precision, recall, _ = precision_recall_curve(
        real_targets,
        probabilities
    )

    ap = average_precision_score(
        real_targets,
        probabilities
    )

    sns.lineplot(
        x=recall,
        y=precision,
        ax=ax,
        linewidth=2.2,
        label=f"Average Precision = {ap:.3f}"
    )

    ax.axhline(
        real_targets.mean(),
        linestyle="--",
        color="gray",
        label="Risk prevalence baseline"
    )

    ax.set_title(stage)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="best")

fig.suptitle(
    "Precision-Recall Curves on Real Student Data",
    fontsize=16,
    y=1.01
)

save_figure(
    fig,
    "05_real_data_precision_recall_curves.png"
)

# Probability Calibration on Real Data

The framework's main output is a probability of academic risk.

Therefore, it is important to determine whether probabilities produced by
synthetic trained models remain meaningful on real students.

Calibration curves compare predicted probabilities with actual risk
frequencies.

A well calibrated model should follow the diagonal reference line.

The evaluation is descriptive at this stage: no real data recalibration is
performed before constructing these curves.

In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 11)
)

for ax, stage in zip(axes.flat, STAGE_ORDER):

    probabilities = frozen_probabilities[stage]

    observed_fraction, mean_prediction = calibration_curve(
        real_targets,
        probabilities,
        n_bins=8,
        strategy="quantile"
    )

    sns.lineplot(
        x=mean_prediction,
        y=observed_fraction,
        marker="o",
        linewidth=2,
        ax=ax,
        label="Observed calibration"
    )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="gray",
        label="Perfect calibration"
    )

    brier = brier_score_loss(
        real_targets,
        probabilities
    )

    ax.set_title(f"{stage}\nBrier Score = {brier:.3f}")
    ax.set_xlabel("Mean Predicted Academic Risk")
    ax.set_ylabel("Observed Academic Risk")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="best")

fig.suptitle(
    "Real Data Probability Calibration",
    fontsize=16,
    y=1.01
)

save_figure(
    fig,
    "06_real_data_calibration_curves.png"
)

# Confusion Matrix Analysis

Confusion matrices are generated for the selected model at each checkpoint

The 4 outcomes are:

- True Negative: correctly identified as not at risk
- False Positive: incorrectly identified as at risk
- False Negative: at-risk student missed by the model
- True Positive: correctly identified as at risk

False negatives are particularly important for the proposed early warning
framework because these students may not receive timely support 

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(12, 11)
)

for ax, stage in zip(axes.flat, STAGE_ORDER):

    probabilities = frozen_probabilities[stage]

    predictions = (
        probabilities >= THRESHOLD
    ).astype(int)

    matrix = confusion_matrix(
        real_targets,
        predictions,
        labels=[0, 1]
    )

    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=["Not At Risk", "At Risk"],
        yticklabels=["Not At Risk", "At Risk"],
        ax=ax
    )

    ax.set_title(stage)
    ax.set_xlabel("Predicted Outcome")
    ax.set_ylabel("Actual Outcome")

fig.suptitle(
    "Confusion Matrices for Real Student Risk Predictions",
    fontsize=16,
    y=1.01
)

save_figure(
    fig,
    "07_real_data_confusion_matrices.png"
)

# Performance Across Real Cohorts

The model's performance is evaluated separately for each real academic year

This analysis helps identify whether predictive performance varies between
cohorts, potentially indicating changes in:

- The student population
- Curriculum
- Assessment records
- Admission patterns
- Academic risk prevalence

Differences between cohorts are descriptive and do not establish their cause

In [ ]:
yearly_metrics = frozen_metrics_df[
    frozen_metrics_df["cohort"] != "All Real Cohorts"
].copy()

yearly_metrics["entry_year"] = (
    yearly_metrics["cohort"].astype(int)
)

yearly_metrics.to_csv(
    METRIC_DIR / "04_performance_by_real_cohort.csv",
    index=False
)

auc_pivot = yearly_metrics.pivot(
    index="stage",
    columns="entry_year",
    values="roc_auc"
).reindex(STAGE_ORDER)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    auc_pivot,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title("ROC AUC by Academic Checkpoint and Real Cohort")
ax.set_xlabel("Entry Year")
ax.set_ylabel("Prediction Checkpoint")

save_figure(
    fig,
    "08_roc_auc_by_real_cohort.png"
)


# Additional year-by-year performance visualisation
fig, ax = plt.subplots(figsize=(11, 6))

sns.lineplot(
    data=yearly_metrics,
    x="entry_year",
    y="recall",
    hue="stage",
    hue_order=STAGE_ORDER,
    marker="o",
    linewidth=2,
    ax=ax
)

ax.set_title("Recall Across Real Academic Cohorts")
ax.set_xlabel("Entry Year")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.set_xticks(sorted(yearly_metrics["entry_year"].unique()))
ax.legend(title="Checkpoint", bbox_to_anchor=(1.02, 1))

save_figure(
    fig,
    "09_recall_across_real_cohorts.png"
)

# Synthetic Versus Real Performance Comparison

This section compares the performance reported during the synthetic Phase 1
experiment against the frozen model's performance on real institutional data

The same checkpoint and metric definitions are used

The main comparison includes:

- ROC AUC
- Recall
- Precision
- F1 Score
- Brier Score

A reduction in performance would suggest that patterns learned from the
synthetic data do not transfer fully to the real observations

The comparison is descriptive because the synthetic and real cohorts are not
identical samples, and the synthetic generation process may have used
historical cohort information

In [ ]:
PHASE1_RESULT_CANDIDATES = [
    PHASE1_RESULTS_DIR / "calibrated_stage_winner_performance.csv",
    PHASE1_RESULTS_DIR / "best_model_at_each_stage.csv",
    PHASE1_RESULTS_DIR / "best_model_by_stage.csv"
]

phase1_result_path = next(
    (p for p in PHASE1_RESULT_CANDIDATES if p.is_file()),
    None
)

synthetic_real_comparison = pd.DataFrame()

if phase1_result_path is not None:

    phase1_results = pd.read_csv(phase1_result_path)

    required_comparison_columns = {
        "stage", "test_roc_auc", "test_recall",
        "test_precision", "test_f1"
    }

    if required_comparison_columns.issubset(
        phase1_results.columns
    ):

        # Keep one selected result per checkpoint.
        phase1_results = phase1_results.drop_duplicates("stage")

        comparison_columns = [
            "test_roc_auc",
            "test_recall",
            "test_precision",
            "test_f1"
        ]

        if "test_brier" in phase1_results:
            comparison_columns.append("test_brier")

        synthetic_results_long = (
            phase1_results[
                ["stage"] + comparison_columns
            ]
            .melt(
                id_vars="stage",
                var_name="metric",
                value_name="value"
            )
        )

        synthetic_results_long["metric"] = (
            synthetic_results_long["metric"]
            .str.replace("test_", "", regex=False)
        )

        synthetic_results_long["dataset"] = (
            "Synthetic Holdout (Phase 1)"
        )

        real_results_long = (
            frozen_overall_df[
                ["stage", "roc_auc", "recall",
                 "precision", "f1", "brier"]
            ]
            .melt(
                id_vars="stage",
                var_name="metric",
                value_name="value"
            )
        )

        real_results_long["dataset"] = (
            "Real Cohorts (Phase 2)"
        )

        synthetic_real_comparison = pd.concat(
            [
                synthetic_results_long,
                real_results_long
            ],
            ignore_index=True
        )

        synthetic_real_comparison.to_csv(
            METRIC_DIR / "05_synthetic_vs_real_comparison.csv",
            index=False
        )

        display(
            synthetic_real_comparison.pivot_table(
                index=["stage", "metric"],
                columns="dataset",
                values="value"
            ).round(4)
        )

        plot_subset = synthetic_real_comparison[
            synthetic_real_comparison["metric"] == "roc_auc"
        ].copy()

        fig, ax = plt.subplots(figsize=(12, 6))

        sns.barplot(
            data=plot_subset,
            x="stage",
            y="value",
            hue="dataset",
            order=STAGE_ORDER,
            ax=ax
        )

        ax.set_title(
            "ROC AUC: Synthetic Holdout vs Real Cohort Evaluation"
        )
        ax.set_xlabel("Prediction Checkpoint")
        ax.set_ylabel("ROC AUC")
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", rotation=15)
        ax.legend(title="Dataset")

        save_figure(
            fig,
            "10_synthetic_vs_real_roc_auc.png"
        )

    else:
        print(
            "Phase 1 file was found, but it does not contain "
            "the required test metrics."
        )

else:
    print(
        "Phase 1 metrics CSV was not found. "
        "Real-data evaluation is complete, but the "
        "synthetic-versus-real comparison requires "
        "the saved Phase 1 metrics file."
    )

# Uncertainty Assessment

Bootstrap resampling is applied to estimate uncertainty in the observed
real data performance of the frozen synthetic models

The following metrics are evaluated:

- ROC AUC
- Recall
- Precision
- F1 Score
- Brier Score

The resulting 95% confidence intervals are calculated separately for:

1. All available real records
2. The latest real cohort, reserved for the later refinement comparison

The intervals describe uncertainty from resampling students in the
evaluation sample

They do not account fully for differences between
academic years or uncertainty in the faculty rule labels

In [ ]:
def bootstrap_confidence_intervals(
    y_true,
    probabilities,
    n_bootstrap=N_BOOTSTRAP,
    random_state=SEED
):

    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    probabilities = np.asarray(probabilities)

    metric_names = [
        "roc_auc",
        "recall",
        "precision",
        "f1",
        "brier"
    ]

    samples = {
        metric: [] for metric in metric_names
    }

    n = len(y_true)

    for _ in range(n_bootstrap):

        indices = rng.integers(
            0, n, size=n
        )

        sample_y = y_true[indices]
        sample_p = probabilities[indices]

        if len(np.unique(sample_y)) < 2:
            continue

        result = evaluate_model(
            sample_y,
            sample_p
        )

        for metric in metric_names:
            samples[metric].append(result[metric])

    rows = []

    for metric in metric_names:

        values = np.asarray(samples[metric])

        rows.append({
            "metric": metric,
            "estimate": float(np.mean(values)),
            "lower_95": float(np.percentile(values, 2.5)),
            "upper_95": float(np.percentile(values, 97.5))
        })

    return pd.DataFrame(rows)


# Latest real cohort is reserved for the refinement holdout.
REAL_HOLDOUT_YEAR = int(aligned["entry_year"].max())

holdout_mask = (
    aligned["entry_year"].to_numpy() == REAL_HOLDOUT_YEAR
)

uncertainty_frames = []

for stage in STAGE_ORDER:

    for scope, mask in [
        ("All Real Cohorts", np.ones(len(aligned), dtype=bool)),
        (f"{REAL_HOLDOUT_YEAR} Holdout", holdout_mask)
    ]:

        intervals = bootstrap_confidence_intervals(
            real_targets[mask],
            frozen_probabilities[stage][mask]
        )

        intervals.insert(0, "stage", stage)
        intervals.insert(1, "evaluation_scope", scope)

        uncertainty_frames.append(intervals)


uncertainty_df = pd.concat(
    uncertainty_frames,
    ignore_index=True
)

uncertainty_df.to_csv(
    METRIC_DIR / "06_real_performance_bootstrap_intervals.csv",
    index=False
)

display(
    uncertainty_df.round(4)
)

# Visualising Uncertainty Across the Academic Timeline

ROC AUC estimates are displayed with 95% bootstrap confidence intervals.

These results indicate whether later predictions provide more reliable
discrimination between at risk and non risk students.

Wide confidence intervals should be interpreted cautiously, especially
for smaller real cohorts.

In [ ]:
auc_ci = uncertainty_df[
    (uncertainty_df["metric"] == "roc_auc")
    & (uncertainty_df["evaluation_scope"] == "All Real Cohorts")
].copy()

auc_ci["stage"] = pd.Categorical(
    auc_ci["stage"],
    categories=STAGE_ORDER,
    ordered=True
)

auc_ci = auc_ci.sort_values("stage")

fig, ax = plt.subplots(figsize=(11, 6))

sns.pointplot(
    data=auc_ci,
    x="stage",
    y="estimate",
    color=sns.color_palette("deep")[0],
    errorbar=None,
    markers="o",
    ax=ax
)

for i, row in enumerate(auc_ci.itertuples()):

    ax.errorbar(
        i,
        row.estimate,
        yerr=[
            [row.estimate - row.lower_95],
            [row.upper_95 - row.estimate]
        ],
        fmt="none",
        capsize=5,
        color="black"
    )

ax.set_title(
    "External ROC AUC Across Checkpoints with 95% Confidence Intervals"
)

ax.set_xlabel("Prediction Checkpoint")
ax.set_ylabel("ROC AUC")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)

save_figure(
    fig,
    "11_real_auc_bootstrap_uncertainty.png"
)

# Multi Stage Academic Risk Trajectories

A core requirement of the proposed framework is that student risk is
reassessed when new academic information becomes available.

Each student has already received 4 probabilities from the frozen
synthetic models.

This section transforms those predictions into a longitudinal dataset,
allowing the evolution of risk to be visualised.

Predicted risk is compared between students who ultimately met and did not
meet the faculty progression requirements.

In [ ]:
trajectory_frames = []

for stage in STAGE_ORDER:

    slug = STAGE_SLUGS[stage]

    temporary = prediction_df[
        ["student_id", "entry_year", "academic_risk"]
    ].copy()

    temporary["stage"] = stage

    temporary["risk_probability"] = (
        prediction_df[
            f"{slug}_risk_probability"
        ]
    )

    temporary["actual_group"] = (
        temporary["academic_risk"]
        .map({
            0: "Ultimately Not At Risk",
            1: "Ultimately At Risk"
        })
    )

    trajectory_frames.append(temporary)


trajectory_df = pd.concat(
    trajectory_frames,
    ignore_index=True
)

trajectory_df["stage"] = pd.Categorical(
    trajectory_df["stage"],
    categories=STAGE_ORDER,
    ordered=True
)

trajectory_df.to_csv(
    PREDICTION_DIR / "02_real_student_longitudinal_risk.csv",
    index=False
)

fig, ax = plt.subplots(figsize=(11, 6))

sns.lineplot(
    data=trajectory_df,
    x="stage",
    y="risk_probability",
    hue="actual_group",
    estimator="mean",
    errorbar=("ci", 95),
    marker="o",
    linewidth=2.5,
    ax=ax
)

ax.set_title(
    "Evolution of Predicted Academic Risk on Real Data"
)

ax.set_xlabel("Prediction Checkpoint")
ax.set_ylabel("Mean Predicted Academic Risk")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)
ax.legend(title="Observed Final Outcome")

save_figure(
    fig,
    "12_real_academic_risk_trajectory.png"
)

# Risk Probability Distributions

The predicted risk distributions of at risk and non risk students are
compared at the final pre examination checkpoint

Greater separation is generally desirable, but probability calibration and
recall should also be considered 

An individual student can still be incorrectly classified even when the
overall distributions are well separated

In [ ]:
final_stage = STAGE_ORDER[-1]

final_probability_df = pd.DataFrame({
    "risk_probability": frozen_probabilities[final_stage],
    "academic_risk": real_targets
})

final_probability_df["actual_group"] = (
    final_probability_df["academic_risk"].map({
        0: "Not At Risk",
        1: "At Risk"
    })
)

fig, ax = plt.subplots(figsize=(11, 6))

sns.histplot(
    data=final_probability_df,
    x="risk_probability",
    hue="actual_group",
    bins=20,
    stat="density",
    common_norm=False,
    element="step",
    multiple="layer",
    alpha=0.35,
    ax=ax
)

ax.axvline(
    THRESHOLD,
    color="black",
    linestyle="--",
    label="Classification threshold"
)

ax.set_title(
    f"Real Student Predicted Risk Distribution — {final_stage}"
)

ax.set_xlabel("Predicted Academic Risk Probability")
ax.set_ylabel("Density")
ax.set_xlim(0, 1)

save_figure(
    fig,
    "13_real_risk_probability_distribution.png"
)

# Descriptive Fairness and Subgroup Performance

The model's performance is examined across selected student subgroups.

The analysis uses:

- Gender
- School quintile.

The primary measures are:

- Recall;
- False negative rate
- False positive rate
- Brier score

Groups with fewer than 20 students are excluded from the reported
subgroup metrics to avoid highly unstable estimates

These statistics indicate potential performance disparities. They do not
establish discrimination or explain why disparities exist

Race and other sensitive fields are not introduced as new model predictors

In [ ]:
fairness_rows = []

audit_groups = [
    col for col in ["gender", "quintile"]
    if col in aligned.columns
]

for stage in STAGE_ORDER:

    probabilities = frozen_probabilities[stage]

    for group_column in audit_groups:

        groups = (
            aligned[group_column]
            .astype("string")
            .fillna("Missing")
        )

        for group_value in groups.unique():

            mask = (
                groups.to_numpy() == group_value
            )

            n_group = int(mask.sum())

            if n_group < MIN_SUBGROUP_N:
                continue

            y_group = real_targets[mask]
            p_group = probabilities[mask]

            result = evaluate_model(y_group, p_group)

            fp = result["false_positive"]
            tn = result["true_negative"]
            fn = result["false_negative"]
            tp = result["true_positive"]

            fairness_rows.append({
                "stage": stage,
                "group_variable": group_column,
                "group": str(group_value),
                "n_students": n_group,
                "recall": result["recall"],
                "false_negative_rate": (
                    fn / (fn + tp)
                    if (fn + tp) > 0 else np.nan
                ),
                "false_positive_rate": (
                    fp / (fp + tn)
                    if (fp + tn) > 0 else np.nan
                ),
                "brier": result["brier"],
                "roc_auc": result["roc_auc"]
            })


fairness_df = pd.DataFrame(fairness_rows)

fairness_df.to_csv(
    METRIC_DIR / "07_subgroup_fairness_metrics.csv",
    index=False
)

display(fairness_df.round(4))


# Plot gender-based false-negative rates at the last checkpoint,
# if enough group observations are available.
if not fairness_df.empty:

    fairness_plot = fairness_df[
        (fairness_df["stage"] == STAGE_ORDER[-1])
        & (fairness_df["group_variable"] == "gender")
    ].copy()

    if not fairness_plot.empty:

        fig, ax = plt.subplots(figsize=(9, 5))

        sns.barplot(
            data=fairness_plot,
            x="group",
            y="false_negative_rate",
            color=sns.color_palette("deep")[1],
            ax=ax
        )

        ax.set_title(
            "False-Negative Rate by Gender Final Prediction Checkpoint"
        )
        ax.set_xlabel("Gender")
        ax.set_ylabel("False-Negative Rate")
        ax.set_ylim(0, 1)

        save_figure(
            fig,
            "14_gender_false_negative_rate.png"
        )

# Real Data Refinement Strategy

The original synthetic trained models have now been evaluated without
modification.

The next step examines whether incorporating real student information
improves predictive performance.

2 refinement approaches are investigated:

1. **Probability recalibration**
   - Keep the synthetic trained classifier frozen.
   - Learn a new probability calibration mapping from earlier real cohorts.

2. **Real data model refitting**
   - Preserve the selected Phase 1 model architecture and hyperparameters.
   - Fit the model again using earlier real cohorts.
   - Recalibrate using cohort aware cross validation.

For this dataset:

- Development cohorts: 2021–2023
- Final real holdout: 2024

The final holdout is never used to fit model parameters or choose whether
recalibration should be applied.

Refitted models are treated as exploratory candidates until they receive
further independent validation.

In [ ]:

# REAL DEVELOPMENT / HOLDOUT SPLIT


REAL_HOLDOUT_YEAR = int(aligned["entry_year"].max())

development_mask = (
    aligned["entry_year"].to_numpy() < REAL_HOLDOUT_YEAR
)

holdout_mask = (
    aligned["entry_year"].to_numpy() == REAL_HOLDOUT_YEAR
)

development_years = sorted(
    aligned.loc[development_mask, "entry_year"].unique()
)

if len(development_years) < 2:
    raise ValueError(
        "At least two development cohorts are required "
        "for cohort-aware Phase 2 refinement."
    )

y_dev = real_targets[development_mask]
y_holdout = real_targets[holdout_mask]

development_groups = (
    aligned.loc[development_mask, "entry_year"]
    .to_numpy()
)


print("PHASE 2 REFINEMENT SPLIT")


print("Development years:", development_years)
print("Real holdout year:", REAL_HOLDOUT_YEAR)

print("Development students:", len(y_dev))
print("Holdout students:", len(y_holdout))

print(
    "Development risk rate:",
    round(y_dev.mean() * 100, 2), "%"
)

print(
    "Holdout risk rate:",
    round(y_holdout.mean() * 100, 2), "%"
)

# Probability Recalibration Using Real Development Cohorts

Probability recalibration adjusts the interpretation of predictions without
changing the underlying synthetic trained classifier

A logistic calibration model is fitted to the log-odds of the frozen model's
predicted probabilities

The recalibration procedure:

1. Obtains frozen predictions for the real development cohorts
2. Uses GroupKFold with academic year as the grouping variable
3. Estimates the Brier score before and after recalibration
4. Decides whether recalibration improves probability quality
5. Fits the final calibration mapping using development cohorts only

The final 2024 cohort remains untouched during this decision

In [ ]:
def logit_transform(probabilities):
    """
    Convert probability values into log-odds for logistic calibration.
    """

    p = np.asarray(probabilities, dtype=float)

    p = np.clip(
        p,
        1e-6,
        1 - 1e-6
    )

    return np.log(
        p / (1 - p)
    ).reshape(-1, 1)


REAL_CALIBRATION_C = 1.0
MIN_BRIER_IMPROVEMENT = 0.002

real_calibrators = {}
recalibrated_holdout_probabilities = {}
selected_refinement_strategy = {}

calibration_decision_rows = []


for stage in STAGE_ORDER:

    print(f"\nRecalibrating: {stage}")

    ----
    # Predictions from the original frozen model
    ----

    p_dev = frozen_probabilities[stage][development_mask]
    p_holdout = frozen_probabilities[stage][holdout_mask]

    ----
    # Cohort-aware calibration cross-validation
    ----

    cv = GroupKFold(
        n_splits=min(
            3,
            len(development_years)
        )
    )

    source_brier_scores = []
    calibrated_brier_scores = []
    validation_sizes = []

    for train_idx, val_idx in cv.split(
        logit_transform(p_dev),
        y_dev,
        groups=development_groups
    ):

        fold_calibrator = LogisticRegression(
            C=REAL_CALIBRATION_C,
            max_iter=2000,
            random_state=SEED
        )

        fold_calibrator.fit(
            logit_transform(p_dev[train_idx]),
            y_dev[train_idx]
        )

        calibrated_validation = (
            fold_calibrator
            .predict_proba(
                logit_transform(p_dev[val_idx])
            )[:, 1]
        )

        original_brier = brier_score_loss(
            y_dev[val_idx],
            p_dev[val_idx]
        )

        calibrated_brier = brier_score_loss(
            y_dev[val_idx],
            calibrated_validation
        )

        source_brier_scores.append(original_brier)
        calibrated_brier_scores.append(calibrated_brier)
        validation_sizes.append(len(val_idx))


    mean_source_brier = np.average(
        source_brier_scores,
        weights=validation_sizes
    )

    mean_calibrated_brier = np.average(
        calibrated_brier_scores,
        weights=validation_sizes
    )

    ----
    # Fit the final calibration map on development data only
    ----

    calibrator = LogisticRegression(
        C=REAL_CALIBRATION_C,
        max_iter=2000,
        random_state=SEED
    )

    calibrator.fit(
        logit_transform(p_dev),
        y_dev
    )

    calibration_slope = float(
        calibrator.coef_[0, 0]
    )

    # Do not accept a calibration mapping that reverses
    # the direction of the original risk ranking.
    use_recalibration = bool(
        (calibration_slope > 0)
        and (
            mean_calibrated_brier
            <= mean_source_brier - MIN_BRIER_IMPROVEMENT
        )
    )

    strategy = (
        "Recalibrated synthetic"
        if use_recalibration
        else "Frozen synthetic"
    )

    selected_refinement_strategy[stage] = strategy

    real_calibrators[stage] = calibrator

    ----
    # Apply to 2024 only AFTER development-based selection
    ----

    recalibrated_holdout = (
        calibrator
        .predict_proba(
            logit_transform(p_holdout)
        )[:, 1]
    )

    recalibrated_holdout_probabilities[stage] = (
        recalibrated_holdout
    )

    calibration_decision_rows.append({
        "stage": stage,
        "development_years": str(development_years),
        "source_cv_brier": mean_source_brier,
        "recalibrated_cv_brier": mean_calibrated_brier,
        "cv_brier_improvement": (
            mean_source_brier - mean_calibrated_brier
        ),
        "calibration_slope": calibration_slope,
        "selected_strategy": strategy,
        "holdout_used_for_selection": False
    })

    # Save the real-data probability calibrator.
    joblib.dump(
        calibrator,
        MODEL_DIR / (
            f"real_probability_calibrator_{STAGE_SLUGS[stage]}.joblib"
        )
    )

    print(
        f"Original CV Brier:     {mean_source_brier:.4f}"
    )
    print(
        f"Recalibrated CV Brier: {mean_calibrated_brier:.4f}"
    )
    print(
        f"Selected strategy:     {strategy}"
    )


calibration_decisions_df = pd.DataFrame(
    calibration_decision_rows
)

calibration_decisions_df.to_csv(
    METRIC_DIR / "08_real_calibration_selection.csv",
    index=False
)

display(
    calibration_decisions_df.round(4)
)

# Refitting Selected Architectures on Real Data

The selected Phase 1 model architecture at each checkpoint is refitted
using the 2021–2023 real development cohorts

The same feature definitions and hyperparameters are retained

Preprocessing is learned again using the real development data rather
than the synthetic data

Probability calibration is performed using cohort grouped cross validation

The resulting models are evaluated on the real 2024 holdout cohort

This is a real data refit, not continued weight level training of the
synthetic model

The refitted models are saved as experimental candidates. They are not
automatically selected for deployment based on 2024 performance

In [ ]:
real_refit_models = {}
real_refit_holdout_probabilities = {}
refit_errors = []


if RUN_REAL_REFIT:

    for stage in STAGE_ORDER:

        print(f"\nRefitting selected architecture: {stage}")

        features = STAGE_FEATURES[stage]

        X_dev = (
            aligned.loc[development_mask, features]
            .copy()
        )

        X_holdout = (
            aligned.loc[holdout_mask, features]
            .copy()
        )

        frozen_model = source_models[stage]

        
        # Extract the selected Phase 1 estimator architecture
        

        if isinstance(
            frozen_model,
            CalibratedClassifierCV
        ):
            original_estimator = frozen_model.estimator
        else:
            original_estimator = frozen_model

        base_estimator = clone(
            original_estimator
        )

        
        # Cohort-aware cross-validation
        

        cv = GroupKFold(
            n_splits=min(
                3,
                len(development_years)
            )
        )

        cv_splits = list(
            cv.split(
                X_dev,
                y_dev,
                groups=development_groups
            )
        )

        refitted_model = CalibratedClassifierCV(
            estimator=base_estimator,
            method="sigmoid",
            cv=cv_splits,
            n_jobs=1
        )

        
        # Train exclusively on real development cohorts
        

        try:

            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")

                refitted_model.fit(
                    X_dev,
                    y_dev
                )

            # Save warnings for reproducibility and diagnosis.
            warning_messages = sorted(set(
                str(w.message) for w in captured
            ))

            if warning_messages:
                warning_path = (
                    REPORT_DIR
                    / f"refit_warnings_{STAGE_SLUGS[stage]}.txt"
                )

                warning_path.write_text(
                    "\n".join(warning_messages),
                    encoding="utf-8"
                )

            probabilities = (
                refitted_model
                .predict_proba(X_holdout)[:, 1]
            )

            real_refit_models[stage] = refitted_model

            real_refit_holdout_probabilities[stage] = (
                probabilities
            )

            joblib.dump(
                refitted_model,
                MODEL_DIR / (
                    f"real_refit_{STAGE_SLUGS[stage]}.joblib"
                )
            )

            result = evaluate_model(
                y_holdout,
                probabilities
            )

            print(
                f"Holdout ROC AUC: {result['roc_auc']:.4f}"
            )
            print(
                f"Holdout Recall:  {result['recall']:.4f}"
            )
            print(
                f"Holdout F1:      {result['f1']:.4f}"
            )
            print(
                f"Holdout Brier:   {result['brier']:.4f}"
            )

        except Exception as exc:

            refit_errors.append({
                "stage": stage,
                "error_type": type(exc).__name__,
                "error": str(exc)
            })

            print(
                f"Real refit could not be completed at {stage}: "
                f"{type(exc).__name__}: {exc}"
            )


pd.DataFrame(
    refit_errors,
    columns=["stage", "error_type", "error"]
).to_csv(
    REPORT_DIR / "real_refit_errors.csv",
    index=False
)

# Final Real Cohort Holdout Comparison

The final real cohort is used to compare three approaches:

1. Frozen synthetic model
2. Synthetic model with real-data probability recalibration
3. Model architecture refitted on real development data

All approaches are evaluated on the same previously untouched 2024 student
records.

The comparison includes:

- ROC AUC
- Recall
- Precision
- F1 score
- Accuracy
- Brier score
- Log loss

The 2024 results are an unbiased holdout assessment of the specified
candidate procedures, provided no model or strategy was chosen using
these results

The test cohort must not be used to retrospectively choose which refitted
model to deploy

In [ ]:
holdout_comparison_rows = []

holdout_prediction_df = aligned.loc[
    holdout_mask,
    ["student_id", "entry_year", "academic_risk"]
].copy()

for stage in STAGE_ORDER:

    slug = STAGE_SLUGS[stage]

    candidate_probabilities = {
        "Frozen synthetic":
            frozen_probabilities[stage][holdout_mask],

        "Recalibrated synthetic":
            recalibrated_holdout_probabilities[stage]
    }

    if stage in real_refit_holdout_probabilities:
        candidate_probabilities["Real-data refit"] = (
            real_refit_holdout_probabilities[stage]
        )

    for version, probabilities in candidate_probabilities.items():

        metrics = evaluate_model(
            y_holdout,
            probabilities
        )

        holdout_comparison_rows.append({
            "stage": stage,
            "model_version": version,
            "holdout_year": REAL_HOLDOUT_YEAR,
            **metrics
        })

        version_slug = (
            version.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        holdout_prediction_df[
            f"{slug}_{version_slug}_probability"
        ] = probabilities


holdout_comparison_df = pd.DataFrame(
    holdout_comparison_rows
)

holdout_comparison_df["stage"] = pd.Categorical(
    holdout_comparison_df["stage"],
    categories=STAGE_ORDER,
    ordered=True
)

holdout_comparison_df = holdout_comparison_df.sort_values(
    ["stage", "model_version"]
)

display(
    holdout_comparison_df[
        [
            "stage",
            "model_version",
            "n_students",
            "roc_auc",
            "recall",
            "precision",
            "f1",
            "brier",
            "log_loss"
        ]
    ].round(4)
)

holdout_comparison_df.to_csv(
    METRIC_DIR / "09_phase2_final_holdout_comparison.csv",
    index=False
)

holdout_prediction_df.to_csv(
    PREDICTION_DIR / "03_2024_holdout_model_comparisons.csv",
    index=False
)

# Visual Comparison of Frozen and Refined Models

The final holdout evaluation is visualised by model version and academic
checkpoint

ROC AUC shows discrimination ability, while Brier score shows probability
quality

For ROC AUC, higher is better

For Brier score, lower is better

These figures allow the effectiveness of refinement to be assessed without
confusing the initial frozen model results with the later experimental
refitting results

In [ ]:
fig, axes = plt.subplots(
    1, 2,
    figsize=(15, 6)
)

sns.barplot(
    data=holdout_comparison_df,
    x="stage",
    y="roc_auc",
    hue="model_version",
    ax=axes[0]
)

axes[0].set_title("Real Holdout ROC AUC")
axes[0].set_xlabel("Prediction Checkpoint")
axes[0].set_ylabel("ROC AUC")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend(title="Model Version", fontsize=8)

sns.barplot(
    data=holdout_comparison_df,
    x="stage",
    y="brier",
    hue="model_version",
    ax=axes[1]
)

axes[1].set_title("Real Holdout Brier Score")
axes[1].set_xlabel("Prediction Checkpoint")
axes[1].set_ylabel("Brier Score (Lower Is Better)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="Model Version", fontsize=8)

save_figure(
    fig,
    "15_frozen_vs_refined_model_performance.png"
)

# Assess Changes in Probability Calibration

The final holdout calibration curves compare:

- The original synthetic trained probabilities
- Probabilities recalibrated using earlier real cohorts

The comparison indicates whether recalibration improved agreement between
predicted risk and observed academic risk frequency

Each calibration curve is constructed entirely from the held out real cohort

In [ ]:
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 11)
)

for ax, stage in zip(axes.flat, STAGE_ORDER):

    candidates = {
        "Frozen synthetic":
            frozen_probabilities[stage][holdout_mask],

        "Recalibrated synthetic":
            recalibrated_holdout_probabilities[stage]
    }

    for label, probabilities in candidates.items():

        observed_fraction, mean_prediction = calibration_curve(
            y_holdout,
            probabilities,
            n_bins=5,
            strategy="quantile"
        )

        sns.lineplot(
            x=mean_prediction,
            y=observed_fraction,
            marker="o",
            linewidth=2,
            label=label,
            ax=ax
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="gray",
        label="Perfect calibration"
    )

    ax.set_title(stage)
    ax.set_xlabel("Mean Predicted Risk")
    ax.set_ylabel("Observed Risk")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)

fig.suptitle(
    "Real Holdout: Probability Calibration Before and After Refinement",
    fontsize=16,
    y=1.01
)

save_figure(
    fig,
    "16_recalibration_before_after.png"
)

# Save the Phase 2 Multi-Stage Model Candidate

The final Phase 2 candidate is built using decisions made exclusively from
the development cohorts

At each checkpoint, it uses:

- The original frozen Phase 1 model
- The frozen model followed by a real data probability calibrator

The choice was made using development cohort Brier scores, not final
holdout performance

Real refitted estimators are saved separately as experimental candidates

The saved bundle also contains the feature definitions and configuration
required for later prospective evaluation

The same data alignment procedure must be applied to new cohorts before
prediction 

In [ ]:
phase2_model_bundle = {

    "framework": "Credit-Aware Early Warning Framework",

    "phase": "Phase 2 - Real Cohort Evaluation and Refinement",

    "target": "academic_risk",

    "target_definition": (
        "1 = not readmitted under progression rules; "
        "0 = readmitted under progression rules"
    ),

    "stage_order": STAGE_ORDER,

    "stage_features": STAGE_FEATURES,

    "selected_phase1_models": selected_model_names,

    "frozen_source_models": source_models,

    "real_probability_calibrators": real_calibrators,

    "selected_strategy_by_stage": selected_refinement_strategy,

    "classification_threshold": THRESHOLD,

    "semester_1_modules": S1_MODULES,

    "semester_2_modules": S2_MODULES,

    "real_development_years": [
        int(y) for y in development_years
    ],

    "real_holdout_year": int(REAL_HOLDOUT_YEAR),

    "feature_alignment_note": (
        "Apply the Phase 2 feature-alignment and "
        "temporal validation code before inference."
    ),

    "random_seed": SEED
}


PHASE2_MODEL_PATH = (
    MODEL_DIR / "phase2_multistage_academic_risk_model.joblib"
)

joblib.dump(
    phase2_model_bundle,
    PHASE2_MODEL_PATH
)

print("Saved Phase 2 candidate model:")
print(PHASE2_MODEL_PATH.resolve())

# Load and Test the Phase 2 Model Candidate

The saved model is loaded to verify that its checkpoint specific predictions
can be reproduced

For each checkpoint, the Phase 2 candidate 1st produces a probability
using the frozen synthetic model

Where real data recalibration was selected during development, the
calibration mapping is then applied

The output is a probability and corresponding risk band

In [ ]:
loaded_phase2_bundle = joblib.load(
    PHASE2_MODEL_PATH
)


def predict_phase2_risk(aligned_student_data, stage):

    bundle = loaded_phase2_bundle

    if stage not in bundle["stage_order"]:
        raise ValueError(
            f"Invalid stage: {stage}"
        )

    features = bundle["stage_features"][stage]

    missing = [
        c for c in features
        if c not in aligned_student_data.columns
    ]

    if missing:
        raise ValueError(
            f"Required aligned features missing: {missing}"
        )

    X = aligned_student_data[features].copy()

    # Frozen synthetic model probability
    model = bundle["frozen_source_models"][stage]

    probability = model.predict_proba(X)[:, 1]

    strategy = bundle[
        "selected_strategy_by_stage"
    ][stage]

    # Apply only when selected using real development data.
    if strategy == "Recalibrated synthetic":

        calibrator = bundle[
            "real_probability_calibrators"
        ][stage]

        probability = (
            calibrator
            .predict_proba(
                logit_transform(probability)
            )[:, 1]
        )

    output = pd.DataFrame({
        "academic_risk_probability": probability,
        "risk_band": assign_risk_band(probability),
        "predicted_class": (
            probability >= bundle["classification_threshold"]
        ).astype(int)
    }, index=aligned_student_data.index)

    return output


# Test on a small sample of already aligned real records.
example_students = aligned.iloc[:5].copy()

example_results = predict_phase2_risk(
    example_students,
    stage="2nd Semester"
)

display(example_results)

# Phase 2 Experiment Summary

The final experiment summary records the files, cohorts, feature mappings,
training choices and saved outputs.

This metadata is important for reproducibility and will support the
methodology and results chapters of the dissertation.

The outputs distinguish:

- The original frozen synthetic model;
- The real-data calibration procedure;
- The experimentally refitted real models; and
- The selected Phase 2 candidate.

All figures and metric tables are saved to the designated output folders.

In [ ]:
experiment_metadata = {

    "framework":
        "Credit-Aware Early Warning Framework",

    "experiment":
        "Phase 2 - Real Cohort Evaluation",

    "real_data_filename":
        REAL_DATA_PATH.name,

    "phase1_model_filename":
        PHASE1_MODEL_PATH.name,

    "number_of_real_students":
        int(len(aligned)),

    "real_cohorts": [
        int(y)
        for y in sorted(aligned["entry_year"].unique())
    ],

    "development_years": [
        int(y) for y in development_years
    ],

    "holdout_year":
        int(REAL_HOLDOUT_YEAR),

    "target_definition":
        "Rule-based academic non-readmission",

    "risk_prevalence":
        float(aligned["academic_risk"].mean()),

    "classification_threshold":
        float(THRESHOLD),

    "stage_models":
        selected_model_names,

    "phase2_strategies":
        selected_refinement_strategy,

    "real_refit_successful_stages":
        list(real_refit_models.keys()),

    "unavailable_phase1_feature_columns":
        missing_phase1_columns,

    "model_output":
        str(PHASE2_MODEL_PATH),

    "bootstrap_iterations":
        int(N_BOOTSTRAP),

    "random_seed":
        int(SEED)
}


with open(
    REPORT_DIR / "phase2_experiment_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        experiment_metadata,
        f,
        indent=4,
        default=str
    )


print("=" * 80)
print("PHASE 2 - REAL COHORT EXPERIMENT SUMMARY")
print("=" * 80)

print(f"\nReal students: {len(aligned):,}")
print(
    "Years:",
    sorted(aligned["entry_year"].unique())
)

print(
    "Real academic risk:",
    f"{aligned['academic_risk'].mean() * 100:.2f}%"
)

print(
    "\nDevelopment years:",
    development_years
)

print(
    "Holdout year:",
    REAL_HOLDOUT_YEAR
)

print("\nSelected checkpoint strategies:")

for stage in STAGE_ORDER:

    print(f"\n{stage}")
    print(
        "  Original algorithm:",
        selected_model_names.get(stage)
    )
    print(
        "  Phase 2 strategy:",
        selected_refinement_strategy[stage]
    )

print("\nOutput directories:")

for directory in [
    METRIC_DIR,
    FIGURE_DIR,
    PREDICTION_DIR,
    MODEL_DIR,
    REPORT_DIR
]:
    print(directory.resolve())

print("\nFinal saved candidate:")
print(PHASE2_MODEL_PATH.resolve())

print("=" * 80)